In [2]:
# ==========================================

!pip install pyomo highspy -q
import pyomo.environ as pyo

model = pyo.ConcreteModel()
model.T = pyo.Set(initialize=range(1, 25))

# --- PARAMETERS ---
load_mw = 10.0

def price_rule(model, t):
    if t <= 6 or t >= 22: return 0.05 # Extremely cheap at night
    else: return 0.60 # Extremely expensive during the day
model.price = pyo.Param(model.T, initialize=price_rule)

def solar_rule(model, t):
    if 8 <= t <= 17: return 8.0
    else: return 0.0
model.solar_avail = pyo.Param(model.T, initialize=solar_rule)

solar_capex = 50000
battery_capex = 50000
max_solar_build = 10.0
max_battery_build = 20.0

# --- VARIABLES ---
model.build_solar = pyo.Var(within=pyo.Binary)
model.build_battery = pyo.Var(within=pyo.Binary)

model.solar_used = pyo.Var(model.T, within=pyo.NonNegativeReals)
model.grid_import = pyo.Var(model.T, within=pyo.NonNegativeReals)
model.bess_charge = pyo.Var(model.T, within=pyo.NonNegativeReals)
model.bess_discharge = pyo.Var(model.T, within=pyo.NonNegativeReals)
model.bess_soc = pyo.Var(model.T, within=pyo.NonNegativeReals)

# --- CONSTRAINTS ---
# 1. Generation Limit
model.solar_limit = pyo.Constraint(model.T, rule=lambda m, t: m.solar_used[t] <= m.solar_avail[t] * m.build_solar)

# 2. Battery Capacity Limit
model.bess_limit = pyo.Constraint(model.T, rule=lambda m, t: m.bess_soc[t] <= max_battery_build * m.build_battery)
model.bess_charge_limit = pyo.Constraint(model.T, rule=lambda m, t: m.bess_charge[t] <= 5.0 * m.build_battery)

# 3. Power Balance (FIXED: using 'model' instead of 'm')
def balance_rule(model, t):
    return model.solar_used[t] + model.grid_import[t] + model.bess_discharge[t] == load_mw + model.bess_charge[t]
model.balance = pyo.Constraint(model.T, rule=balance_rule)

# 4. Battery SOC (FIXED: using 'model' instead of 'm')
def soc_rule(model, t):
    if t == 1:
        return model.bess_soc[t] == 0 + model.bess_charge[t] - model.bess_discharge[t]
    else:
        return model.bess_soc[t] == model.bess_soc[t-1] + model.bess_charge[t] - model.bess_discharge[t]
model.soc = pyo.Constraint(model.T, rule=soc_rule)

# --- OBJECTIVE ---
model.cost = pyo.Objective(
    expr=sum(model.price[t] * model.grid_import[t] for t in model.T) +
         (model.build_solar * solar_capex / 3650) +
         (model.build_battery * max_battery_build * battery_capex / 3650),
    sense=pyo.minimize
)

# Solve
solver = pyo.SolverFactory('appsi_highs')
solver.solve(model)

print(f"Total Daily Optimization Cost (Including Amortized CAPEX): ${model.cost.expr():,.2f}")
print(f"Did we build Solar? {'YES' if model.build_solar.value > 0.5 else 'NO'}")
print(f"Did we build Battery? {'YES' if model.build_battery.value > 0.5 else 'NO'}")
print("-------------------------------------------------------------------------")
print("Hour | Price | Solar Avail | Solar Used | Grid Import | Battery SOC")
for t in model.T:
    print(f"{t:4} | ${model.price[t]:4.2f} | {model.solar_avail[t]:11.1f} | {model.solar_used[t].value:10.1f} | {model.grid_import[t].value:11.1f} | {model.bess_soc[t].value:11.1f}")


Total Daily Optimization Cost (Including Amortized CAPEX): $54.00
Did we build Solar? NO
Did we build Battery? NO
-------------------------------------------------------------------------
Hour | Price | Solar Avail | Solar Used | Grid Import | Battery SOC
   1 | $0.10 |         0.0 |        0.0 |        10.0 |         0.0
   2 | $0.10 |         0.0 |        0.0 |        10.0 |         0.0
   3 | $0.10 |         0.0 |        0.0 |        10.0 |         0.0
   4 | $0.10 |         0.0 |        0.0 |        10.0 |         0.0
   5 | $0.10 |         0.0 |        0.0 |        10.0 |         0.0
   6 | $0.10 |         0.0 |        0.0 |        10.0 |         0.0
   7 | $0.30 |         0.0 |        0.0 |        10.0 |         0.0
   8 | $0.30 |         8.0 |        0.0 |        10.0 |         0.0
   9 | $0.30 |         8.0 |        0.0 |        10.0 |         0.0
  10 | $0.30 |         8.0 |        0.0 |        10.0 |         0.0
  11 | $0.30 |         8.0 |        0.0 |        10.0 |         

In [3]:

# ==========================================
# Week 5 - SCENARIO B: High Peak Pricing
# Shows that when grid prices are high, the model builds the Microgrid.
# ==========================================

model = pyo.ConcreteModel()
model.T = pyo.Set(initialize=range(1, 25))

# --- PARAMETERS ---
load_mw = 10.0

# MODIFIED: High peak price to force microgrid build
def price_rule(model, t):
    if t <= 6 or t >= 22: return 0.05 # Cheap at night
    else: return 0.60 # Very expensive during the day
model.price = pyo.Param(model.T, initialize=price_rule)

def solar_rule(model, t):
    if 8 <= t <= 17: return 8.0
    else: return 0.0
model.solar_avail = pyo.Param(model.T, initialize=solar_rule)

# MODIFIED: Cheaper CAPEX to make it viable
solar_capex = 50000
battery_capex = 50000
max_solar_build = 10.0
max_battery_build = 20.0

# --- VARIABLES ---
model.build_solar = pyo.Var(within=pyo.Binary)
model.build_battery = pyo.Var(within=pyo.Binary)

model.solar_used = pyo.Var(model.T, within=pyo.NonNegativeReals)
model.grid_import = pyo.Var(model.T, within=pyo.NonNegativeReals)
model.bess_charge = pyo.Var(model.T, within=pyo.NonNegativeReals)
model.bess_discharge = pyo.Var(model.T, within=pyo.NonNegativeReals)
model.bess_soc = pyo.Var(model.T, within=pyo.NonNegativeReals)

# --- CONSTRAINTS ---
model.solar_limit = pyo.Constraint(model.T, rule=lambda m, t: m.solar_used[t] <= m.solar_avail[t] * m.build_solar)
model.bess_limit = pyo.Constraint(model.T, rule=lambda m, t: m.bess_soc[t] <= max_battery_build * m.build_battery)
model.bess_charge_limit = pyo.Constraint(model.T, rule=lambda m, t: m.bess_charge[t] <= 5.0 * m.build_battery)

def balance_rule(model, t):
    return model.solar_used[t] + model.grid_import[t] + model.bess_discharge[t] == load_mw + model.bess_charge[t]
model.balance = pyo.Constraint(model.T, rule=balance_rule)

def soc_rule(model, t):
    if t == 1:
        return model.bess_soc[t] == 0 + model.bess_charge[t] - model.bess_discharge[t]
    else:
        return model.bess_soc[t] == model.bess_soc[t-1] + model.bess_charge[t] - model.bess_discharge[t]
model.soc = pyo.Constraint(model.T, rule=soc_rule)

# --- OBJECTIVE ---
model.cost = pyo.Objective(
    expr=sum(model.price[t] * model.grid_import[t] for t in model.T) +
         (model.build_solar * solar_capex / 3650) +
         (model.build_battery * max_battery_build * battery_capex / 3650),
    sense=pyo.minimize
)

# Solve
solver = pyo.SolverFactory('appsi_highs')
solver.solve(model)

print(f"SCENARIO B Cost: ${model.cost.expr():,.2f}")
print(f"Did we build Solar? {'YES' if model.build_solar.value > 0.5 else 'NO'}")
print(f"Did we build Battery? {'YES' if model.build_battery.value > 0.5 else 'NO'}")
print("-------------------------------------------------------------------------")
print("Hour | Price | Solar Avail | Solar Used | Grid Import | Battery SOC")
for t in model.T:



_IncompleteInputError: incomplete input (3024399595.py, line 73)

In [4]:
# ==========================================

!pip install pyomo highspy -q
import pyomo.environ as pyo

model = pyo.ConcreteModel()
model.T = pyo.Set(initialize=range(1, 25))

# --- PARAMETERS ---
load_mw = 10.0

def price_rule(model, t):
    if t <= 6 or t >= 22: return 0.05 # Extremely cheap at night
    else: return 0.60 # Extremely expensive during the day
model.price = pyo.Param(model.T, initialize=price_rule)

def solar_rule(model, t):
    if 8 <= t <= 17: return 8.0
    else: return 0.0
model.solar_avail = pyo.Param(model.T, initialize=solar_rule)

solar_capex = 50000
battery_capex = 50000
max_solar_build = 10.0
max_battery_build = 20.0

# --- VARIABLES ---
model.build_solar = pyo.Var(within=pyo.Binary)
model.build_battery = pyo.Var(within=pyo.Binary)

model.solar_used = pyo.Var(model.T, within=pyo.NonNegativeReals)
model.grid_import = pyo.Var(model.T, within=pyo.NonNegativeReals)
model.bess_charge = pyo.Var(model.T, within=pyo.NonNegativeReals)
model.bess_discharge = pyo.Var(model.T, within=pyo.NonNegativeReals)
model.bess_soc = pyo.Var(model.T, within=pyo.NonNegativeReals)

# --- CONSTRAINTS ---
# 1. Generation Limit
model.solar_limit = pyo.Constraint(model.T, rule=lambda m, t: m.solar_used[t] <= m.solar_avail[t] * m.build_solar)

# 2. Battery Capacity Limit
model.bess_limit = pyo.Constraint(model.T, rule=lambda m, t: m.bess_soc[t] <= max_battery_build * m.build_battery)
model.bess_charge_limit = pyo.Constraint(model.T, rule=lambda m, t: m.bess_charge[t] <= 5.0 * m.build_battery)

# 3. Power Balance (FIXED: using 'model' instead of 'm')
def balance_rule(model, t):
    return model.solar_used[t] + model.grid_import[t] + model.bess_discharge[t] == load_mw + model.bess_charge[t]
model.balance = pyo.Constraint(model.T, rule=balance_rule)

# 4. Battery SOC (FIXED: using 'model' instead of 'm')
def soc_rule(model, t):
    if t == 1:
        return model.bess_soc[t] == 0 + model.bess_charge[t] - model.bess_discharge[t]
    else:
        return model.bess_soc[t] == model.bess_soc[t-1] + model.bess_charge[t] - model.bess_discharge[t]
model.soc = pyo.Constraint(model.T, rule=soc_rule)

# --- OBJECTIVE ---
model.cost = pyo.Objective(
    expr=sum(model.price[t] * model.grid_import[t] for t in model.T) +
         (model.build_solar * solar_capex / 3650) +
         (model.build_battery * max_battery_build * battery_capex / 3650),
    sense=pyo.minimize
)

# Solve
solver = pyo.SolverFactory('appsi_highs')
solver.solve(model)

print(f"Total Daily Optimization Cost (Including Amortized CAPEX): ${model.cost.expr():,.2f}")
print(f"Did we build Solar? {'YES' if model.build_solar.value > 0.5 else 'NO'}")
print(f"Did we build Battery? {'YES' if model.build_battery.value > 0.5 else 'NO'}")
print("-------------------------------------------------------------------------")
print("Hour | Price | Solar Avail | Solar Used | Grid Import | Battery SOC")
for t in model.T:
    print(f"{t:4} | ${model.price[t]:4.2f} | {model.solar_avail[t]:11.1f} | {model.solar_used[t].value:10.1f} | {model.grid_import[t].value:11.1f} | {model.bess_soc[t].value:11.1f}")


Total Daily Optimization Cost (Including Amortized CAPEX): $60.20
Did we build Solar? YES
Did we build Battery? NO
-------------------------------------------------------------------------
Hour | Price | Solar Avail | Solar Used | Grid Import | Battery SOC
   1 | $0.05 |         0.0 |        0.0 |        10.0 |         0.0
   2 | $0.05 |         0.0 |        0.0 |        10.0 |         0.0
   3 | $0.05 |         0.0 |        0.0 |        10.0 |         0.0
   4 | $0.05 |         0.0 |        0.0 |        10.0 |         0.0
   5 | $0.05 |         0.0 |        0.0 |        10.0 |         0.0
   6 | $0.05 |         0.0 |        0.0 |        10.0 |         0.0
   7 | $0.60 |         0.0 |        0.0 |        10.0 |         0.0
   8 | $0.60 |         8.0 |        8.0 |         2.0 |         0.0
   9 | $0.60 |         8.0 |        8.0 |         2.0 |         0.0
  10 | $0.60 |         8.0 |        8.0 |         2.0 |         0.0
  11 | $0.60 |         8.0 |        8.0 |         2.0 |        